# Native Sparse Attention, part 5: hardware alignment
*Companion notebooks to Yuan et al., 2025, [Native Sparse Attention](https://arxiv.org/abs/2502.11089). Part 5 of 7. (Part 4, the gated layer, comes after the toy-training notebooks; this part only needs the pieces from parts 0-2.)*

**TL;DR.** Skipping most of the attention matrix only helps if the skipped work is what the hardware was waiting for. Yuan et al. report, against Triton FlashAttention-2 on A100s, **9.0x faster forward and 6.0x faster backward at 64k tokens, and an expected 11.6x in decoding**. The decoding number is not a measurement we can reproduce without a GPU, but it follows from simple counting that we **can** reproduce exactly: during decoding a query reads `(s)/d + n*l' + w` tokens instead of `s`, so for `s = 65,536` it reads 5,632 instead of 65,536. We reproduce the paper's Table 4, show the memory cost of keeping three branches' keys and values, emulate the paper's group-centric kernel loop in plain PyTorch and check it against our vectorised code, and say clearly where we stop: **we ran no GPU kernels and measured no GPU speeds.**

<img src="nsa_assets/fig6_speed.png" width="700"/>

*Figure 6 of the paper: Triton NSA kernel vs Triton FlashAttention-2, forward (left) and backward (right). Notice the speedup grows with length (2.1x at 8k to 9.0x at 64k forward; 1.1x to 6.0x backward): the saving is a fraction of tokens, and that fraction improves as the context grows.*

In [1]:
import math, time
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"   # used only where a notebook trains
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "font.size": 10})
CMP_C, SLC_C, WIN_C, FULL_C = "#1c7ed6", "#d9480f", "#2f9e44", "#495057"      # one colour per branch, used in every notebook
NEG = -1e30                                                                     # a finite "minus infinity": a fully masked row gives uniform weights, never NaN

CHECKS = []
def check(name, ok, detail=""):
    """Record + print PASS/FAIL. Every claim in these notebooks is backed by one of these lines."""
    CHECKS.append((name, bool(ok)))
    print(("PASS  " if ok else "FAIL  ") + name + (f"   [{detail}]" if detail else ""))

In [2]:
def full_causal_attention(q, k, v):
    """Ordinary causal attention, with the grouped-query layout NSA uses.
    q: (B, G, H, T, dk)   G key/value groups, H query heads share each group's K and V (GQA)
    k: (B, G, T, dk)      v: (B, G, T, dv)       returns (output (B,G,H,T,dv), weights (B,G,H,T,T))"""
    T, dk = q.shape[-2], q.shape[-1]
    scores = torch.einsum("bghtd,bgsd->bghts", q, k) / math.sqrt(dk)          # how much query t likes key s
    causal = torch.ones(T, T, dtype=torch.bool, device=q.device).tril()        # token t may only look at tokens <= t
    p = scores.masked_fill(~causal, NEG).softmax(-1)                           # each row sums to 1
    return torch.einsum("bghts,bgsd->bghtd", p, v), p

In [3]:
class Compressor(nn.Module):
    """Squeeze every block of `l` consecutive keys (or values) into ONE vector.
    Blocks start every `d` tokens; d < l means neighbouring blocks overlap (paper: l=32, d=16).
    phi = learnable position embedding inside the block -> flatten -> small MLP."""
    def __init__(self, dim, l, d):
        super().__init__(); self.l, self.d = l, d
        self.pos = nn.Parameter(torch.zeros(l, dim))                           # intra-block position encoding
        self.mlp = nn.Sequential(nn.Linear(l * dim, dim), nn.GELU(), nn.Linear(dim, dim))
        nn.init.zeros_(self.mlp[2].weight); nn.init.zeros_(self.mlp[2].bias)   # our choice: start as plain mean-pooling, learn a correction
    def forward(self, x):                                                      # x: (B, G, T, dim)
        blocks = x.unfold(2, self.l, self.d).transpose(-1, -2)                 # (B, G, Nc, l, dim): the overlapping blocks
        return blocks.mean(-2) + self.mlp((blocks + self.pos).flatten(-2))     # (B, G, Nc, dim)   mean-pool + learned phi

def n_compressed(T, l, d): return (T - l) // d + 1

def compressed_valid(T, l, d, device="cpu"):
    """(T, Nc) bool. Compressed block j covers tokens [j*d, j*d + l). Query t may use it only if ALL of its
    tokens are <= t (otherwise the block would leak the future)."""
    j = torch.arange(n_compressed(T, l, d), device=device); t = torch.arange(T, device=device)
    return (j[None, :] * d + l - 1) <= t[:, None]

def compressed_attention(q, k_cmp, v_cmp, valid):
    """q (B,G,H,T,dk) attends to the compressed keys. Returns (output, attention weights p_cmp (B,G,H,T,Nc))."""
    s = torch.einsum("bghtd,bgcd->bghtc", q, k_cmp) / math.sqrt(q.shape[-1])
    p = s.masked_fill(~valid, NEG).softmax(-1) * valid                          # zero out invalid blocks (and empty rows)
    return torch.einsum("bghtc,bgcd->bghtd", p, v_cmp), p

In [4]:
def overlap_matrix(T, l, d, lp, device="cpu"):
    """M[i, j] = number of tokens shared by compressed block i and selection block j (size lp).
    p_slc = p_cmp @ M turns 'which compressed blocks does the query like' into 'which selection blocks look important'."""
    Nc, Ns = n_compressed(T, l, d), T // lp
    M = torch.zeros(Nc, Ns)
    for i in range(Nc):
        a, b = i * d, i * d + l
        for j in range(Ns): M[i, j] = max(0, min(b, (j + 1) * lp) - max(a, j * lp))
    return (M / d).to(device)

def minmax_block_scores(q, k, lp):
    """Quest-style heuristic (NO learned parameters): upper-bound each block's best dot product from per-channel min/max of its keys."""
    B, G, T, dk = k.shape; kb = k.view(B, G, T // lp, lp, dk)
    kmin, kmax = kb.min(3).values, kb.max(3).values                             # (B,G,Ns,dk)
    qq = q.unsqueeze(-2)                                                       # (B,G,H,T,1,dk)
    return torch.maximum(qq * kmin[:, :, None, None], qq * kmax[:, :, None, None]).sum(-1)   # (B,G,H,T,Ns)

def select_blocks(block_score, T, lp, n, n_local=2):
    """block_score (B,G,H,T,Ns) -> indices (B,G,T,n) of the blocks each query will read at full resolution.
    1) add scores over the heads of a group (they must agree because they share one KV cache)  [paper eq. 10]
    2) only blocks that start at or before the query are allowed
    3) the first block and the query's own / previous block are always kept (paper: 1 initial + 2 local)
    4) take the top-n"""
    s = block_score.sum(2)                                                      # (B,G,T,Ns)
    Ns = s.shape[-1]; j = torch.arange(Ns, device=s.device); t = torch.arange(T, device=s.device); tb = t // lp
    valid = j[None, :] * lp <= t[:, None]
    forced = ((j[None, :] == 0) | (j[None, :] == tb[:, None]) | ((j[None, :] == tb[:, None] - 1) & (n_local >= 2))) & valid
    s = torch.where(forced, torch.full_like(s, 1e9), s.masked_fill(~valid, NEG))
    return s.topk(n, dim=-1).indices

def selected_attention(q, k, v, idx, lp):
    """Gather the chosen blocks of K,V for every query and attend to them. Cost per query = n*lp tokens."""
    B, G, T, dk = k.shape; Ns = T // lp; n = idx.shape[-1]
    kb, vb = k.view(B, G, Ns, lp, dk), v.view(B, G, Ns, lp, v.shape[-1])
    bi, gi = torch.arange(B, device=k.device)[:, None, None, None], torch.arange(G, device=k.device)[None, :, None, None]
    ks, vs = kb[bi, gi, idx].reshape(B, G, T, n * lp, dk), vb[bi, gi, idx].reshape(B, G, T, n * lp, -1)   # (B,G,T,n*lp,.)
    pos = (idx[..., None] * lp + torch.arange(lp, device=k.device)).reshape(B, G, T, n * lp)             # real position of every gathered token
    valid = (pos <= torch.arange(T, device=k.device)[None, None, :, None])[:, :, None]                    # (B,G,1,T,n*lp) causal
    s = torch.einsum("bghtd,bgtsd->bghts", q, ks) / math.sqrt(dk)
    p = s.masked_fill(~valid, NEG).softmax(-1) * valid
    return torch.einsum("bghts,bgtsd->bghtd", p, vs)

In [5]:
def window_attention(q, k, v, w):
    """Attend only to the last w tokens (including the current one)."""
    B, G, T, dk = k.shape
    kw = F.pad(k, (0, 0, w - 1, 0)).unfold(2, w, 1).transpose(-1, -2)          # (B,G,T,w,dk): window of every position
    vw = F.pad(v, (0, 0, w - 1, 0)).unfold(2, w, 1).transpose(-1, -2)
    pos = torch.arange(T, device=k.device)[:, None] - (w - 1) + torch.arange(w, device=k.device)[None, :]
    valid = (pos >= 0)[None, None, None]                                         # the left padding is not a real token
    s = torch.einsum("bghtd,bgtwd->bghtw", q, kw) / math.sqrt(dk)
    p = s.masked_fill(~valid, NEG).softmax(-1) * valid
    return torch.einsum("bghtw,bgtwd->bghtd", p, vw)

## Counting what one decoding step reads

Decoding is memory-bound (part 0), so time is roughly proportional to the bytes of keys and values read per generated token. For a cached history of `s` tokens, full attention reads `s` tokens. NSA reads, per KV group:

* **compressed tokens:** about `s / d` (one per block start; the paper's Table 4 uses `s/d`),
* **selected tokens:** `n * l'`,
* **window tokens:** `w`.

The paper's Table 4 lists exactly these totals: 2048, 2560, 3584 and 5632 tokens for contexts of 8k, 16k, 32k and 64k, and the "expected speedup" 4x, 6.4x, 9.1x, 11.6x.

{img("table4_decode_memory.png", 600)}

In [6]:
l, d, lp, n, w = 32, 16, 64, 16, 512                          # paper configuration
paper = {8192: (2048, 4.0), 16384: (2560, 6.4), 32768: (3584, 9.1), 65536: (5632, 11.6)}
print(f"{'context s':>10s} {'compressed':>11s} {'selected':>9s} {'window':>7s} {'NSA reads':>10s} {'paper Table 4':>14s} {'speedup (s / reads)':>20s}")
for s, (tok, sp) in paper.items():
    cmp_, slc, win = s // d, n * lp, w; total = cmp_ + slc + win
    print(f"{s:>10,d} {cmp_:>11,d} {slc:>9,d} {win:>7,d} {total:>10,d} {tok:>14,d} {s / total:>20.1f}x")
    check(f"s={s:,}: our token count equals the paper's Table 4 ({tok:,}) and the speedup rounds to {sp}x", s // d + n * lp + w == tok and round(s / tok, 1) == sp)

 context s  compressed  selected  window  NSA reads  paper Table 4  speedup (s / reads)
     8,192         512     1,024     512      2,048          2,048                  4.0x
PASS  s=8,192: our token count equals the paper's Table 4 (2,048) and the speedup rounds to 4.0x
    16,384       1,024     1,024     512      2,560          2,560                  6.4x
PASS  s=16,384: our token count equals the paper's Table 4 (2,560) and the speedup rounds to 6.4x
    32,768       2,048     1,024     512      3,584          3,584                  9.1x
PASS  s=32,768: our token count equals the paper's Table 4 (3,584) and the speedup rounds to 9.1x
    65,536       4,096     1,024     512      5,632          5,632                 11.6x
PASS  s=65,536: our token count equals the paper's Table 4 (5,632) and the speedup rounds to 11.6x


Four contexts, four exact matches. This is arithmetic, not evidence that the kernel achieves these speedups: it is the paper's *expected* figure, which they obtain by assuming time follows memory traffic. Their measured decode speedup at 64k, 11.6x (Figure 1), lands on the same number.

## What the saved bytes are worth, and what NSA adds back

To make "bytes" concrete, use the paper's model shape: 30 layers, 4 KV groups, `d_k = 192`, `d_v = 128`, bf16 (2 bytes). Our back-of-envelope, one decoded token at 64k context, KV-cache reads only, A100 bandwidth about 2.0 TB/s (datasheet):

In [7]:
layers, groups, dk, dv, nbytes = 30, 4, 192, 128, 2
per_token = (dk + dv) * nbytes                                   # bytes of K and V for one token in one group
s = 65536
full_bytes = s * per_token * groups * layers
nsa_bytes = (s // d + n * lp + w) * per_token * groups * layers
bw = 2.0e12
print(f"full attention reads {full_bytes / 1e9:5.2f} GB of KV cache per decoded token  ->  {1e3 * full_bytes / bw:5.2f} ms at 2 TB/s")
print(f"NSA reads            {nsa_bytes / 1e9:5.2f} GB                                 ->  {1e3 * nsa_bytes / bw:5.2f} ms")
check("byte ratio equals the token ratio", abs(full_bytes / nsa_bytes - 65536 / 5632) < 1e-6)

# The cost side: what must be STORED. Three branches have their own K and V (part 3).
store_full = s * per_token                                       # one K,V per token (full attention)
store_nsa = (s * per_token                                       # selection branch keeps every token's K,V
             + (s // d) * per_token                              # compressed branch: one K,V per block
             + w * per_token)                                    # window branch: only the last w tokens
print(f"\nKV cache per group-layer, full attention: {store_full / 1e6:6.2f} MB    NSA (3 branches): {store_nsa / 1e6:6.2f} MB   (+{100 * (store_nsa / store_full - 1):.1f}%)")
check("keeping separate K,V for three branches costs about 7% extra cache, not 3x", 1.05 < store_nsa / store_full < 1.10, f"+{100 * (store_nsa / store_full - 1):.1f}%")

full attention reads  5.03 GB of KV cache per decoded token  ->   2.52 ms at 2 TB/s
NSA reads             0.43 GB                                 ->   0.22 ms
PASS  byte ratio equals the token ratio

KV cache per group-layer, full attention:  41.94 MB    NSA (3 branches):  44.89 MB   (+7.0%)
PASS  keeping separate K,V for three branches costs about 7% extra cache, not 3x   [+7.0%]


**Benefit and cost in one paragraph.** Reading 11.6x fewer bytes per token is the decode saving. The independent keys and values that part 3 argued for do *not* triple the cache, as one might first fear: the window branch only needs its last 512 tokens and the compressed branch has 16x fewer entries, so our count is about 7% more cache than full attention. (Our arithmetic; the paper does not give a cache-size figure. We also ignore the small extra weights for the compressor and gates.)

## Why contiguous blocks are fast

Part 2 chose **blocks** of 64 tokens rather than individual tokens. GPUs fetch memory in wide, aligned chunks; 64 consecutive tokens fit in a few chunks, while 64 scattered tokens touch 64 different ones. We cannot measure a GPU here. As a small, honest stand-in we time the same operation on this machine's CPU (copy 4,096 rows of 192 floats from a 200 MB array, as 64 contiguous blocks of 64 vs 4,096 random rows). The effect on a GPU is larger and differently shaped; this only shows the direction on a cache-based CPU.

In [8]:
import time
cache = torch.randn(262144, 192)                                  # 262,144 tokens x 192 floats (~200 MB)
rng = np.random.default_rng(0)
starts = rng.integers(0, 262144 // 64, 64) * 64
block_idx = torch.tensor(np.concatenate([np.arange(s0, s0 + 64) for s0 in starts]))   # 64 contiguous blocks of 64 rows
rand_idx = torch.tensor(rng.integers(0, 262144, 4096))                                    # 4,096 random rows
def bench(idx, reps=200):
    cache.index_select(0, idx)                                     # warm-up
    t0 = time.perf_counter()
    for _ in range(reps): cache.index_select(0, idx)
    return (time.perf_counter() - t0) / reps * 1e6
tb, tr = bench(block_idx), bench(rand_idx)
print(f"gather 4096 rows as 64 contiguous blocks: {tb:7.1f} us     as random rows: {tr:7.1f} us     ratio {tr / tb:.2f}x   (CPU, this machine, not a GPU)")
check("the gather is correct: both patterns return the requested rows", torch.equal(cache.index_select(0, block_idx)[:64], cache[starts[0]:starts[0] + 64]))

gather 4096 rows as 64 contiguous blocks:   123.6 us     as random rows:   231.4 us     ratio 1.87x   (CPU, this machine, not a GPU)
PASS  the gather is correct: both patterns return the requested rows


Read the ratio as whatever it came out as on this machine; we assert nothing about its size, because CPU caches behave differently from GPU memory systems, and our `index_select` over single rows is not how a GPU kernel would stream blocks. The mechanism the paper relies on, coalesced loads and tensor-core-sized tiles, is a GPU property we did not test.

## The kernel's loop, emulated in PyTorch

FlashAttention-style kernels never build the score matrix. For one query they stream through blocks of keys and values, keeping a running maximum `m`, a running normaliser `z` and a running weighted sum `acc`, a trick called **online softmax**. NSA's selection kernel (the paper's Figure 3) differs in *what the outer loop is over*:

{img("fig3_kernel.png", 560)}

*Figure 3 of the paper. The outer "grid loop" is over query positions; for each position the kernel loads all `H` query heads of one GQA group (they share the same chosen blocks), then the inner loop streams only the selected KV blocks into fast on-chip memory (SRAM), computes there, and writes the output back to HBM. Notice what it avoids: FlashAttention's usual outer loop over blocks of consecutive queries, which would force every query in the block to wait for the union of everyone's selected blocks.*

Here is that algorithm for one query position, in plain PyTorch, and a check that it gives the same answer as our vectorised `selected_attention`:

In [9]:
def group_centric_selected_attention(q, k, v, idx, lp):
    """Emulation of the NSA selection kernel's loop. q (B,G,H,T,dk); k,v (B,G,T,.); idx (B,G,T,n) selected blocks.
    For each (batch, group, position): hold ALL H query heads, stream the selected blocks one at a time with online softmax."""
    B, G, H, T, dk = q.shape; dv = v.shape[-1]; out = torch.zeros(B, G, H, T, dv)
    for b in range(B):
        for g in range(G):
            for t in range(T):                                         # "grid loop" over query positions
                qt = q[b, g, :, t]                                     # (H, dk): all heads of the group, loaded once
                m = torch.full((H,), -float("inf")); z = torch.zeros(H); acc = torch.zeros(H, dv)
                for blk in idx[b, g, t].tolist():                      # "inner loop": stream the shared selected blocks
                    lo = blk * lp; pos = torch.arange(lo, lo + lp)
                    Kb, Vb = k[b, g, lo:lo + lp], v[b, g, lo:lo + lp]  # one contiguous block of K and V
                    s = (qt @ Kb.T) / math.sqrt(dk)                    # (H, lp)
                    s = s.masked_fill(pos[None, :] > t, -float("inf")) # causal
                    if torch.isinf(s.max()) and s.max() < 0: continue  # nothing visible in this block
                    m_new = torch.maximum(m, s.max(-1).values)
                    scale = torch.exp(m - m_new).nan_to_num(0.0)       # rescale what we have accumulated so far
                    p = torch.exp(s - m_new[:, None])
                    z = z * scale + p.sum(-1); acc = acc * scale[:, None] + p @ Vb; m = m_new
                out[b, g, :, t] = acc / z[:, None]
    return out

torch.manual_seed(0)
B, G, H, T, dk, lp = 1, 2, 4, 48, 16, 8
q, k, v = torch.randn(B, G, H, T, dk), torch.randn(B, G, T, dk), torch.randn(B, G, T, dk)
idx = select_blocks(torch.randn(B, G, H, T, T // lp), T, lp, 3)
ref = selected_attention(q, k, v, idx, lp)
emu = group_centric_selected_attention(q, k, v, idx, lp)
check("the streaming (online-softmax, group-centric) loop equals the vectorised selected attention", torch.allclose(ref, emu, atol=1e-5), f"max diff {(ref - emu).abs().max():.1e}")

PASS  the streaming (online-softmax, group-centric) loop equals the vectorised selected attention   [max diff 3.6e-07]


This validates the *algorithm* (stream the chosen blocks, never materialise scores, share the load across the heads of a group). It says nothing about speed: a Python loop is the opposite of a fast kernel.

## Counting why the loop order matters

Why does the paper loop over *positions* and not over blocks of queries? Suppose a block of 64 consecutive queries each select 16 blocks. A kernel that loads a query block must bring in the **union** of those selections so every query finds its blocks. How big is the union? It depends on how much neighbouring queries agree, which we cannot measure without a trained model, so we simulate the range: identical choices (best case), independent choices (worst case), and one in between.

In [10]:
def union_size(corr, Ns=1024, n=16, n_forced=3, queries=64, trials=200, seed=0):
    """Blocks a query-block-centric kernel must load for 64 consecutive queries, when query scores are
    `corr` correlated across queries (1 = identical choices, 0 = independent). Forced blocks are shared by all."""
    g = torch.Generator().manual_seed(seed); sizes = []
    for _ in range(trials):
        shared = torch.randn(Ns, generator=g); own = torch.randn(queries, Ns, generator=g)
        sc = corr * shared + math.sqrt(1 - corr ** 2) * own
        sc[:, :1] = 1e9                                                                    # initial block always chosen
        sc[:, -2:] = 1e9                                                                   # local blocks always chosen
        sizes.append(len(set(sc.topk(n, dim=-1).indices.flatten().tolist())))
    return float(np.mean(sizes))

print(f"{'agreement between neighbouring queries':>40s} {'blocks loaded per 64 queries':>30s} {'vs. what ONE query needs':>26s}")
rows = {c: union_size(c) for c in (1.0, 0.9, 0.5, 0.0)}
for c, u in rows.items(): print(f"{c:>40.1f} {u:>30.0f} {u / 16:>25.1f}x")
check("identical choices need exactly 16 blocks (best case)", abs(rows[1.0] - 16) < 1e-9)
check("independent choices need many times more blocks than one query uses", rows[0.0] > 10 * 16, f"{rows[0.0] / 16:.0f}x")
check("more agreement means a smaller union", rows[1.0] < rows[0.9] < rows[0.5] < rows[0.0])

  agreement between neighbouring queries   blocks loaded per 64 queries   vs. what ONE query needs


                                     1.0                             16                       1.0x
                                     0.9                            101                       6.3x
                                     0.5                            376                      23.5x
                                     0.0                            575                      35.9x
PASS  identical choices need exactly 16 blocks (best case)
PASS  independent choices need many times more blocks than one query uses   [36x]
PASS  more agreement means a smaller union


So the penalty of a query-block-centric kernel ranges from nothing (if neighbouring queries choose the same blocks) to an order of magnitude or more (if they do not), and where real models sit is an empirical question we did not answer. The group-centric order sidesteps the question: each query loads exactly its own `n` blocks, once, and the `H = 16` heads of the group share that load.

Our accounting of what that buys, per query and per group, at the paper's sizes: bytes loaded `= n * l' * (d_k + d_v) * 2`; FLOPs `= H * 2 * n * l' * (d_k + d_v)`; so **16 FLOPs per byte**.

In [11]:
H, n, lp, dk, dv = 16, 16, 64, 192, 128
bytes_loaded = n * lp * (dk + dv) * 2
flops = H * 2 * n * lp * (dk + dv)
print(f"arithmetic intensity of the group-centric selection loop: {flops / bytes_loaded:.1f} FLOP/byte   (one head alone: {flops / H / bytes_loaded:.1f};  A100 critical intensity ~156)")
check("sharing the load across the group's 16 heads multiplies intensity by 16", abs(flops / bytes_loaded - 16) < 1e-9)

arithmetic intensity of the group-centric selection loop: 16.0 FLOP/byte   (one head alone: 1.0;  A100 critical intensity ~156)
PASS  sharing the load across the group's 16 heads multiplies intensity by 16


By this accounting the selection kernel is about 16x better than a single head would be, yet still below the A100's critical intensity (about 156 FLOP/byte, part 0), so it remains on the memory side of the roofline. The paper describes the design as achieving "near-optimal arithmetic intensity" by removing redundant transfers and balancing work across the GPU's streaming multiprocessors; we did not verify that statement and our accounting only covers the per-query sharing, not any reuse inside the kernel we cannot see.

## Closing thoughts

What we could check without a GPU: the decode-side expected speedups of the paper's Table 4 follow exactly from the token counts; the extra cache for three branches is about 7%, not 3x; the group-centric streaming loop computes the same function as our reference code; and the choice of loop order matters by a factor that depends on how much neighbouring queries agree. What we could not check, and do not claim: the measured 9.0x forward, 6.0x backward and 11.6x decoding speedups in the paper's Figures 1 and 6, which come from Triton kernels on A100 hardware.

**Claims that lack a number or baseline:** (1) the CPU timing is a direction check on the wrong hardware; (2) the cache-overhead and memory-time figures are our arithmetic from the paper's configuration and datasheet bandwidth; (3) the union-size range is a simulation over a correlation parameter, not a measurement on a trained model.

In [12]:
print(f"{sum(ok for _, ok in CHECKS)}/{len(CHECKS)} checks passed")
for n, ok in CHECKS:
    if not ok: print("FAILED:", n)
assert all(ok for _, ok in CHECKS), "some checks failed"

12/12 checks passed
